In [14]:
import numpy as np
import pandas as pd

In [15]:
import os, shutil

In [16]:
train_path='../image-dataset/dogs-vs-cats/train'

In [17]:
animal=[]
img_path=[]
for file in os.listdir(train_path):
    animal.append(str(file.split('.')[0]))
    img_path.append(file)

In [18]:
df=pd.DataFrame({'animal': animal,'img':img_path})

In [19]:
df.head()

,animal,img
0,cat,cat.0.jpg
1,cat,cat.1.jpg
2,cat,cat.10.jpg
3,cat,cat.100.jpg
4,cat,cat.1000.jpg


In [20]:
def convert(value):
    if value=='cat':
        return 1
    else:
        return 0
df['animal']=df['animal'].apply(convert)

In [21]:
df.head()

,animal,img
0,1,cat.0.jpg
1,1,cat.1.jpg
2,1,cat.10.jpg
3,1,cat.100.jpg
4,1,cat.1000.jpg


In [22]:
train_df = df.sample(frac=1,random_state=0).iloc[:20000]
test_df = df.sample(frac=1,random_state=0).iloc[20000:]

In [23]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [24]:
train_datagen=ImageDataGenerator(rescale=1./255,
                                 rotation_range=30,
                                 width_shift_range=0.2,
                                 height_shift_range=0.2,
                                 shear_range=0.2,
                                 zoom_range=0.2,
                                 horizontal_flip=True)

test_datagen=ImageDataGenerator(rescale=1./255)

In [27]:
train_generator = train_datagen.flow_from_dataframe(train_df,
                                                    directory=train_path,
                                                    x_col='img',
                                                    y_col='animal',
                                                    target_size=(128,128),
                                                    class_mode='raw')

test_generator = test_datagen.flow_from_dataframe(test_df,
                                                    directory=train_path,
                                                    x_col='img',
                                                    y_col='animal',
                                                    target_size=(128,128),
                                                  class_mode='raw')

Found 20000 validated image filenames.
Found 5000 validated image filenames.


## feature extraction data aug

In [28]:
from tensorflow import keras
from keras import Sequential
from keras.layers import Dense,Flatten
from keras.applications.vgg16 import VGG16

In [29]:
conv_base=VGG16(
    weights='imagenet',
    include_top=False,
    input_shape=(128,128,3)
)

In [30]:
model = Sequential()

model.add(conv_base)
model.add(Flatten())
model.add(Dense(256,activation='relu'))
model.add(Dense(1,activation='sigmoid'))

In [31]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ vgg16 (Functional)              │ (None, 4, 4, 512)      │    14,714,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 8192)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │     2,097,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           257 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 16,812,353 (64.13 MB)

 Trainable params: 16,812,353 (64.13 MB)

 Non-trainable params: 0 (0.00 B)

In [33]:
conv_base.trainable = False

In [34]:
model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])

In [ ]:
history = model.fit(train_generator,epochs=5,validation_data=test_generator)

c:\Users\moury\Desktop\Shouryagna_Parvam\AI-ML-DL\myenv\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/5
120/625 ━━━━━━━━━━━━━━━━━━━━ 16:24 2s/step - accuracy: 0.7171 - loss: 0.6703

In [ ]:
import matplotlib.pyplot as plt

plt.plot(history.history['accuracy'],color='red',label='train')
plt.plot(history.history['val_accuracy'],color='blue',label='validation')
plt.legend()
plt.show()

In [ ]:
plt.plot(history.history['loss'],color='red',label='train')
plt.plot(history.history['val_loss'],color='blue',label='validation')
plt.legend()
plt.show()